In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
synthesize_purple_glare.py
批量为图像注入淡紫/淡绿光晕 + 色差，用于去紫光模型的数据集构建
==============================================================
依赖:  pip install opencv-python numpy tqdm
用法:  python synthesize_purple_glare.py --src imgs/clean --dst imgs/flare \
        --repeat 3 --max-radius 0.25
"""
import cv2
import os
import random
import argparse
import numpy as np
from tqdm import tqdm
from glob import glob

# ---------- 光晕蒙版 ----------
def make_halo_mask(h, w, center, radius, color, intensity=0.7, sigma_ratio=0.5):
    """
    返回一张 HxWx3 的 float32 蒙版，数值∈[0,1]，叠加后即可产生淡光晕
    """
    yy, xx = np.mgrid[0:h, 0:w]
    dist2 = (xx - center[0])**2 + (yy - center[1])**2
    sigma = (radius * sigma_ratio)**2
    alpha = np.exp(-dist2 / (2 * sigma)) * intensity            # HxW
    mask = np.dstack([alpha * c for c in color])                 # HxWx3
    return mask.astype(np.float32)

# ---------- 色差 ----------
def add_chromatic_aberration(img, max_shift=3):
    """
    随机平移 R/B 通道形成彩边
    """
    h, w = img.shape[:2]
    dx_r, dy_r = random.randint(-max_shift, max_shift), random.randint(-max_shift, max_shift)
    dx_b, dy_b = random.randint(-max_shift, max_shift), random.randint(-max_shift, max_shift)

    def shift_channel(channel, dx, dy):
        M = np.float32([[1, 0, dx], [0, 1, dy]])
        return cv2.warpAffine(channel, M, (w, h), borderMode=cv2.BORDER_REFLECT101)

    b, g, r = cv2.split(img)
    r_shift = shift_channel(r, dx_r, dy_r)
    b_shift = shift_channel(b, dx_b, dy_b)
    return cv2.merge([b_shift, g, r_shift])

# ---------- 主流程 ----------
def synthesize_one(img, max_radius_ratio=0.3, purple_prob=0.7):
    h, w = img.shape[:2]

    # 1) 随机决定光晕参数
    max_r = int(min(h, w) * max_radius_ratio)
    radius = random.randint(max_r // 3, max_r)
    center = (random.randint(radius, w - radius),
              random.randint(radius, h - radius))
    # 紫 or 绿
    if random.random() < purple_prob:
        color = np.array([1.0, 0.0, 1.0])      # BGR: purple
    else:
        color = np.array([0.0, 1.0, 0.0])      # green
    intensity = random.uniform(0.35, 0.8)

    # 2) 构造光晕
    halo = make_halo_mask(h, w, center, radius, color, intensity)

    # 3) 叠加光晕（线性曝光模型）
    img_f = img.astype(np.float32) / 255.0
    flare = np.clip(img_f + halo, 0.0, 1.0)

    # 4) 添加色差
    flare_ca = add_chromatic_aberration((flare * 255).astype(np.uint8))
    return flare_ca

def batch_synthesize(src_dir, dst_dir, repeat=1, max_radius=0.3):
    os.makedirs(dst_dir, exist_ok=True)
    paths = sorted(sum([glob(os.path.join(src_dir, f'*{ext}')) for ext in ('jpg', 'png', 'jpeg')], []))

    for p in tqdm(paths, desc=f'Synth x{repeat}'):
        img = cv2.imread(p, cv2.IMREAD_COLOR)
        if img is None:
            continue
        base = os.path.splitext(os.path.basename(p))[0]
        for k in range(repeat):
            aug = synthesize_one(img, max_radius_ratio=max_radius)
            cv2.imwrite(os.path.join(dst_dir, f'{base}_flare{k+1}.png'), aug)

# ---------- CLI ----------
def parse_args():
    ap = argparse.ArgumentParser()
    ap.add_argument('--src', required=True, help='原始图像文件夹')
    ap.add_argument('--dst', required=True, help='输出文件夹')
    ap.add_argument('--repeat', type=int, default=2, help='每张图重复合成次数')
    ap.add_argument('--max-radius', type=float, default=0.3, help='光晕最大半径占短边比例')
    return ap.parse_args()

if __name__ == '__main__':
    args = parse_args()
    batch_synthesize(args.src, args.dst, args.repeat, args.max_radius)

In [1]:
import cv2
import numpy as np
import os

def add_purple_fringe(img, intensity=0.6, max_width=2):
    """
    模拟光学紫边：在高亮边缘添加偏紫色晕边。
    :param img: 输入RGB图像
    :param intensity: 紫边强度，范围0~1
    :param max_width: 紫边最大宽度，单位为像素
    """
    img = img.astype(np.float32) / 255.0
    gray = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 100, 200)

    # 扩展边缘范围形成边缘带
    kernel = np.ones((max_width, max_width), np.uint8)
    edge_band = cv2.dilate(edges, kernel, iterations=1)

    # 构造紫色mask
    purple_mask = np.zeros_like(img)
    purple_mask[..., 0] = 0.8  # Red
    purple_mask[..., 1] = 0.0  # Green
    purple_mask[..., 2] = 1.0  # Blue

    mask = (edge_band > 0).astype(np.float32)[..., None]
    blended = img * (1 - intensity * mask) + purple_mask * (intensity * mask)

    return np.clip(blended * 255, 0, 255).astype(np.uint8)

In [2]:
def add_glow_fringe(img, threshold=220, glow_radius=15, glow_strength=0.5):
    """
    模拟传感器高光溢出形成的紫光晕
    :param img: 输入RGB图像
    :param threshold: 高亮触发阈值（0~255）
    :param glow_radius: 模糊半径
    :param glow_strength: 紫光强度（0~1）
    """
    img = img.astype(np.float32) / 255.0
    gray = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    highlight_mask = (gray > threshold).astype(np.uint8) * 255

    # 高斯模糊扩散形成光晕
    glow = cv2.GaussianBlur(highlight_mask, (0, 0), glow_radius)
    glow = cv2.normalize(glow.astype(np.float32), None, 0, 1.0, cv2.NORM_MINMAX)

    purple_layer = np.zeros_like(img)
    purple_layer[..., 0] = 0.8  # R
    purple_layer[..., 2] = 1.0  # B

    blend = img * (1 - glow_strength * glow[..., None]) + purple_layer * (glow_strength * glow[..., None])

    return np.clip(blend * 255, 0, 255).astype(np.uint8)

In [3]:
# 读取图像
img = cv2.imread('1.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 加入紫边和光晕
img_fringe = add_purple_fringe(img_rgb, intensity=0.6, max_width=2)
img_glow = add_glow_fringe(img_fringe, threshold=220, glow_radius=10, glow_strength=0.4)

# 保存结果
cv2.imwrite('purple_fringe_output.jpg', cv2.cvtColor(img_glow, cv2.COLOR_RGB2BGR))

True

In [1]:
import cv2
import numpy as np
import random

def add_sparse_purple_fringe(img, intensity=0.2, max_width=1, sparse_ratio=0.1):
    """
    添加稀疏的紫边伪影，适合数据增强使用。
    :param img: 输入RGB图像，np.uint8
    :param intensity: 紫边的颜色强度（建议0.1~0.3）
    :param max_width: 紫边最大宽度（建议1~2）
    :param sparse_ratio: 稀疏比例（0~1），表示有多少比例的边缘点被选中
    """
    img = img.astype(np.float32) / 255.0
    gray = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 100, 200)

    # 稀疏采样边缘
    mask_indices = np.argwhere(edges > 0)
    selected_indices = mask_indices[np.random.choice(
        len(mask_indices), size=int(len(mask_indices) * sparse_ratio), replace=False
    )]

    mask = np.zeros_like(gray, dtype=np.float32)
    for y, x in selected_indices:
        mask[max(0, y - max_width):y + max_width + 1, max(0, x - max_width):x + max_width + 1] = 1.0

    mask = cv2.GaussianBlur(mask, (3, 3), 0)

    # 紫色图层（偏红蓝）
    purple = np.zeros_like(img)
    purple[..., 0] = 0.6  # R
    purple[..., 1] = 0.0  # G
    purple[..., 2] = 0.8  # B

    # 融合
    mask = mask[..., None]
    result = img * (1 - intensity * mask) + purple * (intensity * mask)

    return np.clip(result * 255, 0, 255).astype(np.uint8)

In [2]:
# 读取原图像
img = cv2.imread('1.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 添加稀疏紫边
fringe_img = add_sparse_purple_fringe(img_rgb, intensity=0.2, max_width=1, sparse_ratio=0.2)

# 保存或展示
cv2.imwrite('fringe_slight_0.2.jpg', cv2.cvtColor(fringe_img, cv2.COLOR_RGB2BGR))

True

紫晕

In [ ]:
import cv2, numpy as np, argparse, os

def add_purple_flare(src_path, dst_path,
                     thresh=230, intensity=0.7,
                     dilate_iter=2, sigma_mask=15, sigma_purple=50):
    img = cv2.imread(src_path)
    h, w = img.shape[:2]

    # 1) 亮区检测
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = (gray > thresh).astype(np.uint8)          # 二值掩膜
    mask = cv2.dilate(mask, None, iterations=dilate_iter)
    mask = cv2.GaussianBlur(mask.astype(float), (0,0), sigmaX=sigma_mask)
    mask = np.clip(mask, 0, 1)

    # 2) 紫色条纹（可调为条带状、径向或渐变）
    purple = np.zeros_like(img, dtype=float)
    purple[...,0] = 255   # B
    purple[...,1] =  60   # G
    purple[...,2] = 180   # R
    purple = cv2.GaussianBlur(purple, (0,0), sigmaX=sigma_purple)

    # 3) 混合
    out = img.astype(float)
    out = out*(1-mask[...,None]) + purple*mask[...,None]*intensity
    out = np.clip(out, 0, 255).astype(np.uint8)

    cv2.imwrite(dst_path, out)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("src", help="input image")
    parser.add_argument("dst", help="output image")
    parser.add_argument("--thresh", type=int, default=230)
    parser.add_argument("--intensity", type=float, default=0.7)
    args = parser.parse_args()
    add_purple_flare(args.src, args.dst, args.thresh, args.intensity)